# pyraingen Evaluation: Stochastic Rainfall Generation in Python

**Companion notebook to:** _Stochastic Rainfall Generation with pyraingen: A Practitioner's Evaluation_

This notebook documents a real evaluation of [pyraingen](https://pypi.org/project/pyraingen/) 1.0.0 (the current release on PyPI as of September 2026) run in a Linux sandbox. Some cells below execute live (pure numpy, safe to re-run). Others document real issues found during testing, shown as captured transcripts rather than live cells — re-running the environment-fix or the 44MB example-data copy live isn't something this notebook should do on every clone.

**Bottom line up front:** the package installs, but getting past `import` requires pinning `numpy<2` yourself (its own metadata doesn't), and its core daily/subdaily rainfall generator is not runnable on Linux at all via a plain `pip install` — see Section 2.

In [1]:
import numpy as np
print('numpy:', np.__version__)
import pyraingen
print('pyraingen: installed OK (this top-level import is lazy -- it does not by itself pull in the broken submodules below)')

numpy: 2.4.6
pyraingen: installed OK (this top-level import is lazy -- it does not by itself pull in the broken submodules below)


## 1. The dependency pin problem

`pip show -f pyraingen` / the package's own `METADATA` declares:

```
Requires-Dist: numpy (>=1.23.5)
Requires-Dist: pandas (==1.5.3)
Requires-Dist: xarray (==2023.01.0)
```

`numpy` has no upper bound, but `pandas` and `xarray` are hard-pinned to versions that predate numpy 2.0 (released mid-2024) and whose compiled C extensions are ABI-incompatible with it. A plain `pip install pyraingen` on any environment where pip resolves a current numpy reproduces this real, captured error:

```
>>> from pyraingen.regionaliseddailysim import regionaliseddailysim
...
File ".../pandas/_libs/interval.pyx", line 1, in init pandas._libs.interval
ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject
```

Confirmed in a completely clean, fresh virtualenv (`python -m venv`, then `pip install pyraingen` with no other packages pre-installed) — not just a quirk of one messy environment. **The fix:** pin numpy yourself before installing: `pip install "numpy<2" pyraingen`. With that pin, `pandas==1.5.3` and `xarray==2023.01.0` both import cleanly.

## 2. The bigger blocker: no Linux build of the core simulation engine

Even with the numpy pin fixed, the package's actual daily rainfall generator fails:

```
>>> from pyraingen.regionaliseddailysim import regionaliseddailysim
...
File ".../pyraingen/regionaliseddailysim.py", line 8, in <module>
    from .fortran_daily import regionalised_dailyT4
ImportError: cannot import name 'regionalised_dailyT4' from 'pyraingen.fortran_daily'
```

The reason is visible directly in the installed package's file listing — `pyraingen/fortran_daily/` ships:

```
regionalised_dailyT.cp38-win_amd64.pyd
regionalised_dailyT4.cp38-win_amd64.pyd
regionalised_dailyT.for
regionalised_dailyT4.for
```

`.pyd` files are compiled Windows extension modules — these two are built for **CPython 3.8 on 64-bit Windows specifically**. There is no `.so` (Linux) or `.dylib` (macOS) build anywhere in the package, and no source-compilation fallback happens automatically on `pip install`. The Fortran source (`.for`) is present, so compiling it yourself via `f2py` is possible in principle — not attempted here.

Practically: **`regionaliseddailysim` — the actual stochastic daily rainfall generator, the core of what the package does — cannot be run at all via a plain `pip install pyraingen` on Linux or macOS**, which is where most cloud/CI/HPC-based hydrology work happens today. `regionalisedsubdailysim` has a code path (`genSeqOption` 0/1/2) that can work from existing daily data without calling the Fortran daily generator; `genSeqOption=3` ("no daily or sub-daily data available — generate the daily series first") hits the same wall.

## 3. What does actually run: `computeIFD`

One piece is pure numpy with no compiled or pinned-package dependency: `pyraingen.computeifd.computeIFD`, described in its own docstring as an internal helper "designed to be used inside the IFD Conditioning code" (`ifdcond`, one of the package's three advertised main functions) rather than something end users call directly. It takes a pre-built 6-minute rainfall array — shape `(240, nDays, nSimulations)` — and a matching years vector, and is meant to return an annual-maximum table by duration.

The array below is a **deliberately simple, clearly-synthetic** 6-minute series — sparse random rainfall bursts, not a real disaggregated sequence — used only to exercise the function's actual contract, not to represent real rainfall behaviour anywhere.

In [2]:
from pyraingen.computeifd import computeIFD

rng = np.random.default_rng(20260901)
n_days, n_sims, n_records_per_day = 365 * 10, 3, 240

rainfall = np.zeros((n_records_per_day, n_days, n_sims))
for sim in range(n_sims):
    n_bursts = 300
    burst_days = rng.integers(0, n_days, n_bursts)
    burst_starts = rng.integers(0, n_records_per_day - 10, n_bursts)
    burst_depths = rng.gamma(shape=2.0, scale=3.0, size=n_bursts)
    for d, s, depth in zip(burst_days, burst_starts, burst_depths):
        dur = rng.integers(1, 8)
        rainfall[s:s+dur, d, sim] += depth / dur

years_vector = np.repeat(np.arange(2016, 2026), 365)
ifd_durations = [6, 30, 60, 360, 1440]  # minutes

print(f'Input: rainfall{rainfall.shape} (240 x nDays x nSims), requested {len(ifd_durations)} durations, nSims={n_sims}')
IFD = computeIFD(rainfall, years_vector, ifd_durations)
print(f'Output shape: {IFD.shape}')
print(f'Docstring says output should be (nYears, nSimulations, nIFDDurations) = ({len(set(years_vector))}, {n_sims}, {len(ifd_durations)})')

Input: rainfall(240, 3650, 3) (240 x nDays x nSims), requested 5 durations, nSims=3
Output shape: (10, 240, 5)
Docstring says output should be (nYears, nSimulations, nIFDDurations) = (10, 3, 5)


The output's second axis comes back as **240**, not the 3 simulations requested. Looking at the source (`inspect.getsource`), the output array is allocated as:

```python
IFD = np.zeros((len(yearsUnique), np.size(rainfallSeries, axis=0), len(ifdDurations)))
```

`np.size(rainfallSeries, axis=0)` is the 240 six-minute-slots-per-day count — not `nSimulations` (axis 2 per the function's own documented input shape). The fill loop then does `np.max(np.max(..., axis=2), axis=1)` — collapsing across *both* the simulation axis and the within-year day axis — before assigning into a 240-long slot. That doesn't produce a per-simulation annual-maximum-by-duration table; it produces something else, and the function's own source contains a telling loose end in its process comments:

```python
# The general process for each simulation is:
#   -) Aggregate up from 6 minute if required.
#   -) Extract the annual maximum series
#   -) ?
```

That trailing `?` is the original author's own comment, not mine. I'd read this as: the annual-maximum extraction step looks unresolved even to whoever wrote it, and the actual output shape doesn't match what the docstring promises. I wouldn't trust this specific function's return values for a real project without independently re-deriving the annual maxima from the aggregated series myself.

## 4. The real target-IFD format

`get_example_data()` (once worked around — see the companion article for that bug too) bundles a real example `targetifds.csv`: 6 rows x 4 columns, matching `ifdcond`'s documented defaults `AEP=[63.2, 50, 20, 10, 5, 2]` (6 values) x `TargetIFDdurationsEst=[30, 60, 360, 720]` minutes (4 values). `ifdcond` itself implements "Algorithm from Fitsum et al. (2016) for Constraining continuous rainfall simulations for derived design flood estimation" per its own docstring — a real, legitimate IFD-conditioning method. Getting to the point of actually calling it end-to-end needs subdaily simulation output this environment couldn't produce (Section 2).

## 5. Summary

| Piece | Status |
|---|---|
| `pip install pyraingen` | Installs, but needs a manual `numpy<2` pin to import cleanly |
| `regionaliseddailysim` (core daily generator) | Not runnable on Linux/macOS — Windows/CPython-3.8-only compiled binaries, no fallback |
| `regionalisedsubdailysim` | Partially blocked — depends on `genSeqOption`; option 3 hits the same wall |
| `ifdcond` (IFD conditioning) | Imports fine once numpy is pinned; needs upstream subdaily output as input |
| `computeIFD` (internal helper) | Runs, but its return shape looks inconsistent with its own docstring |

See the companion article for the full write-up and what I'd actually recommend to a colleague considering this package today.